In [1]:
import sys
if '..' not in sys.path:
    sys.path.append("..")

In [2]:
from transformers import AutoTokenizer
from rl_retrieval.utils import all_facts_found
from rl_retrieval.feedback import GroundTruthFeedback
from rl_retrieval.llm_feedback import LLMJudge, ExactMatchFeedback, MutualInformationFeedback, StepwiseFactRelevanceFeedback

from rl_retrieval.retrieval_envs.qa_retrieval_env import QARetrievalEnv, SimpleEnvAdapter
from rl_retrieval.policy import RNDPolicy, OraclePolicy
from rl_retrieval.utils import all_facts_found
from dataloaders.localsets.musique import RetrievalMusique
from dataloaders.localsets.hotpotqa import RetrievalHotPotQA
from dataloaders.localsets.babilong import RetrievalBabilong

from dataloaders.globalset import PATHS
import numpy as np

In [3]:
"../" + PATHS['musique']

'../data_sources/musique'

In [4]:
seed = 42
split = 'eval'
num_samples = 5
#model_name = 'Qwen/Qwen-32B'
model_name = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
#tokenizer here are used only to estimate length of samples in datasets
dataset = RetrievalHotPotQA(
    path="../" + PATHS['hotpotqa'], #as we run this from notebooks/ subfloder
    tokenizer=tokenizer, length=-1,
    min_context_len=0, max_context_len=1e7,
    type='any', anno_type='any', split=split, seed=seed
)

dataset = SimpleEnvAdapter(dataset)
reward_model = None
termination_func = None

#fb_model = GroundTruthReward(per_fact_reward=0.05, completion_reward=1.)
#fb_model = LLM_Judge_Reward('Qwen/Qwen2.5-32B-Instruct', completion_reward=1.)
#fb_model = Exact_Match('Qwen/Qwen2.5-32B-Instruct', completion_reward=2.)
#fb_model = Information_Based_Reward('Qwen/Qwen2.5-32B-Instruct', completion_coeff=1.)
terminated = truncated = False
states = []
rewards = []
agent = OraclePolicy()

HotPotQA load:   0%|          | 0/7405 [00:00<?, ?it/s]

In [5]:
#!pip install --upgrade transformers

In [6]:
dataset[7]

{'id': '5a881d2355429938390d3eeb',
 'question': ' If You Ever Get Lonely was covered by what Lyric Street Records-affiliated band',
 'answer': 'Love and Theft',
 'chunks_texts': ['Sarah Buxton Sarah Buxton (born July 3, 1980) is an American country music singer, formerly signed to the independent Lyric Street Records.  Between 2006 and 2008, she issued three singles from an extended play titled "Almost My Record", in addition to co-writing her song "Stupid Boy", which was later recorded by Keith Urban.  She released her self-titled debut album in early 2010, led off by the Top 25 single "Outside My Window," shortly before Lyric Street Records closed.  Shortly afterward, she began performing with Jedd Hughes as the short-lived duo Buxton Hughes before forming Skyline Motel.',
  'Love and Theft (duo) Love and Theft is an American country music group founded by Stephen Barker Liles, Eric Gunderson, and Brian Bandas, all three of whom alternated as lead singers and acoustic guiarists.  Sig

In [13]:
fb_model = StepwiseFactRelevaynceFeedback(model_name, per_fact_reward=0.5)
env = QARetrievalEnv(fb_model, max_steps=2)
num_samples = 1
for i in range(num_samples):
    step = 0
    terminated = truncated = False
    print(f"\n################## START EPISODE #{i} ####################")
    print(f'=========== Step #{step} ===========')
    obs, info = env.reset(dataset[i])
    print(f"Question: {obs['question']}?")
    print('Relevant chunks:')
    for j in info['sf_idx']:
        print(f"Chunk #{j}:\n {obs['chunks'][j]}\n")


    while not (terminated or truncated):
        # generally agent should avoid using info but these are just examples:
        chunk_idx = agent.act(obs, info)
        print(f'Agent selected chunks: {chunk_idx}')
        obs, reward, terminated, truncated, info = env.step(chunk_idx)
        print(f'reward - {reward}')
        print(f"last_action - {info['last_action']}")
        print(f"last_chunks - {obs['chunks'][info['last_action'][0]]}")
        step += 1
        print(f'Reward: {reward:.2f}, terminated: {terminated}, truncated: {truncated}')
        print(f"chunk mask: {obs['chunks_mask']}")
        print('Retrieved chunks:\n', "\n".join(obs['retrieved_chunks']), "\n")
        print(f'=========== Step #{step} ===========')
        # log episode steps(obs, reward, terminated, truncated, info)
        # do some training

    reward_model.completed = (terminated or truncated)
    completion_reward = reward_model.reward(obs, info)
    print(f"completion_reward - {completion_reward}")

    print(f"################## END EPISODE #{i} ####################")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]


################## START EPISODE #0 ####################
=========== Step #0 ===========
Question: VIVA Media AG changed it's name in 2004. What does their new acronym stand for?
Relevant chunks:
Chunk #5:
 VIVA Media VIVA Media GmbH (until 2004 "VIVA Media AG") is a music television network originating from Germany.  It was founded for broadcast of VIVA Germany as VIVA Media AG in 1993 and has been owned by their original concurrent Viacom, the parent company of MTV, since 2004.  Viva channels exist in some European countries; the first spin-offs were launched in Poland and Switzerland in 2000.

Chunk #7:
 Gesellschaft mit beschränkter Haftung A Gesellschaft mit beschränkter Haftung (] , abbreviated GmbH ] and also GesmbH in Austria) is a type of legal entity very common in Germany, Austria, Switzerland (where it is equivalent to a S.à r.l.) and Liechtenstein.  In the United States, the equivalent type of entity is the limited liability company (LLC).  The name of the GmbH form empha

NotImplementedError: Could not run 'flash_attn::_flash_attn_forward' with arguments from the 'CPU' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'flash_attn::_flash_attn_forward' is only available for these backends: [CUDA, Meta, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradHIP, AutogradXLA, AutogradMPS, AutogradIPU, AutogradXPU, AutogradHPU, AutogradVE, AutogradLazy, AutogradMTIA, AutogradPrivateUse1, AutogradPrivateUse2, AutogradPrivateUse3, AutogradMeta, AutogradNestedTensor, Tracer, AutocastCPU, AutocastXPU, AutocastMPS, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor, FuncTorchVmapMode, Batched, VmapMode, FuncTorchGrad